Install deps

In [1]:
!pip install google-adk requests

setup imports and api key

In [2]:
import requests
import os
from typing import Optional, List, Dict
from google.adk import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-fab96dd167ae"  # replace with your actual project ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

GOOGLE_MAPS_API_KEY = "YOUR_API_KEY_HERE"

Get lat/long from verbal location

In [3]:
def get_lat_lon(location: str) -> Dict[str, float]:
    """Convert a place name to latitude and longitude using Google Maps Geocoding API.

    Args:
        location: A place name or address (e.g., "Denver, CO").

    Returns:
        A dictionary with 'lat' and 'lng' keys, or an 'error' key if not found.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": location, "key": GOOGLE_MAPS_API_KEY}
    resp = requests.get(url, params=params)
    data = resp.json()
    if data["status"] == "OK":
        loc = data["results"][0]["geometry"]["location"]
        return {"lat": loc["lat"], "lng": loc["lng"]}
    return {"error": f"Geocoding failed: {data['status']}"}

Weather API function

In [4]:
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Retrieve the extended weather forecast from the National Weather Service API.

    Args:
        lat: Latitude of the location.
        lon: Longitude of the location.

    Returns:
        A list of forecast periods with 'name', 'temperature', and 'forecast' keys,
        or None if the request fails.
    """
    headers = {"User-Agent": "weather-agent-app"}
    # Step 1: Get the forecast endpoint for this location
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    resp = requests.get(points_url, headers=headers)
    if resp.status_code != 200:
        return None
    forecast_url = resp.json()["properties"]["forecast"]

    # Step 2: Get the actual forecast
    resp = requests.get(forecast_url, headers=headers)
    if resp.status_code != 200:
        return None
    periods = resp.json()["properties"]["periods"]
    return [
        {
            "name": p["name"],
            "temperature": f"{p['temperature']}°{p['temperatureUnit']}",
            "forecast": p["detailedForecast"],
        }
        for p in periods
    ]

Weather agent instructions

In [9]:
weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.0-flash-lite",
    description="An agent that provides weather forecasts for any US location.",
    instruction="""You are a helpful weather assistant. When the user asks about the weather:
1. Ask for their location if not provided.
2. Use get_lat_lon to convert the location to coordinates.
3. Use get_extended_weather_forecast to retrieve the forecast.
4. Summarize the forecast in a friendly, readable way.""",
    tools=[get_lat_lon, get_extended_weather_forecast],
)

Agent run

In [6]:
async def run_agent(agent, user_message: str) -> str:
    """Run the agent with a single user message and return the final response."""
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=agent.name, user_id="test_user"
    )
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)

    content = types.Content(
        role="user", parts=[types.Part.from_text(text=user_message)]
    )
    final_response = ""
    async for event in runner.run_async(
        user_id="test_user", session_id=session.id, new_message=content
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final_response = event.content.parts[0].text
    return final_response

Test geocoding function

In [7]:
print(get_lat_lon("Denver, CO"))

{'lat': 39.7392358, 'lng': -104.990251}


Test cities

In [11]:
test_cities = [
  "What's the weather in Denver, Colorado?",
  "Give me the forecast for Miami, Florida",
  "What's the weather like in Seattle, Washington?",
]

for query in test_cities:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    result = await run_agent(weather_agent, query)
    print(result)


Query: What's the weather in Denver, Colorado?
The weather in Denver, Colorado today will be mostly sunny with a high near 68 degrees Fahrenheit. Tonight, there's a chance of rain and snow with a low around 32 degrees. On Friday, expect snow with a high near 37 degrees. The weekend will be sunny with highs in the 50s and 60s.

Query: Give me the forecast for Miami, Florida
The forecast for Miami, Florida is:

*   **Today:** Mostly sunny with a high of 79°F. There's a slight chance of showers and thunderstorms after 5 PM.
*   **Tonight:** Partly cloudy with a low of 74°F. There's a slight chance of showers and thunderstorms before midnight.
*   **Friday:** Partly sunny with a high of 79°F. There's a slight chance of rain showers between 7 AM and 1 PM, and a slight chance of showers and thunderstorms.
*   **Friday Night:** Mostly cloudy with a low of 74°F. There's a slight chance of showers and thunderstorms.
*   **Saturday:** Partly sunny with a high of 80°F. There's a slight chance of